# Smart Request Router - System 1 in Colab

Runs the **System 1** of the router (SetFit + escalation policy) on the published model, on the free CPU runtime:
no server, no Docker, no keys. Messages it should not decide are shown as *escalations* with the reason
(System 2, the LLM agent, is not part of this notebook).

Code: https://github.com/Leito2/smart-request-router - Model: https://huggingface.co/WhiteUniversal/smart-request-router-setfit

In [ ]:
%%capture
!pip install -q "routercore[setfit] @ git+https://github.com/Leito2/smart-request-router.git@main#subdirectory=packages/routercore" pyyaml huggingface_hub

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download

from routercore.adapters.policy_config import load_policy
from routercore.adapters.setfit_model import SetFitRouter
from routercore.application.contracts import Escalation, InboundMessage
from routercore.application.route_messages import RouteMessages

model_dir = Path(snapshot_download("WhiteUniversal/smart-request-router-setfit"))   # ~1 GB, once
policy, threshold_set = load_policy(model_dir / "policy.yaml")
router = RouteMessages(SetFitRouter.load(model_dir), policy, threshold_set)
print("thresholds:", threshold_set)

In [ ]:
def route(*texts: str) -> None:
    messages = [InboundMessage(message_id=f"m{i}", customer_id="colab", channel="chat", text=t, lang=None,
                               ts_sent_ns=0) for i, t in enumerate(texts)]
    for text, event in zip(texts, router(messages)):
        if isinstance(event, Escalation):
            print(f"ESCALATES to System 2 (reason: {event.reason})  <- {text}")
        else:
            print(f"System 1 -> {event.route:10s} priority={event.priority:6s} confidence={event.confidence:.2f}  <- {text}")


route(
    "What is the exchange rate fee when I pay abroad?",
    "Hola, quiero saber cuánto cuesta una transferencia internacional",
    "My card was stolen and someone just spent 400 euros with it, please block everything now",
    "Why was my payment pay_8842 blocked? I need it to go through today",
    "Can you help?",
)

## Try your own

System 1 answers when it is confident and nothing looks urgent; otherwise it escalates (the safety net
for urgent messages is deliberately aggressive: see the README, "Selective prediction").

In [ ]:
route("I want to close my account", "my transfer has not arrived after 5 days")